# Energy Partitioning Simulation: Hydrotectonic Model v2.3

**Purpose:** Numerically model how gravitational potential energy partitions during Stage 2 hydraulic collapse.

**Author:** James (JD) Longmire  
**Date:** 2025-12-17  
**Model Version:** 2.3 (Objection-Response Edition)

---

## Critic's Challenge

"You claim 99% of PE goes to seismic/plastic/residual, but none of these are calculated. Show where 10²⁵ J actually goes."

This notebook provides a time-stepping numerical simulation to track energy flow during Stage 2.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from dataclasses import dataclass

# Set up plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['font.size'] = 12

## 1. Model Parameters

These parameters come from the main model document (Appendices A and B).

In [ ]:
@dataclass
class ModelParameters:
    """Physical parameters for the Hydrotectonic Model."""
    
    # Block parameters
    n_blocks: int = 10                    # Number of continental blocks
    block_length: float = 1000e3          # m (1000 km)
    block_width: float = 800e3            # m (800 km)
    block_thickness: float = 30e3         # m (30 km)
    rho_crust: float = 2700               # kg/m³
    
    # Derived block properties
    @property
    def block_volume(self):
        return self.block_length * self.block_width * self.block_thickness
    
    @property
    def block_mass(self):
        return self.block_volume * self.rho_crust
    
    @property
    def base_area(self):
        return self.block_length * self.block_width
    
    # Stress parameters
    g: float = 9.8                        # m/s²
    detachment_depth: float = 15e3        # m (15 km)
    pore_pressure_ratio: float = 0.99     # λ = P_pore / σ_lithostatic
    friction_coeff: float = 0.01          # μ (reduced friction)
    
    @property
    def sigma_lithostatic(self):
        """Lithostatic stress at detachment depth (Pa)."""
        return self.rho_crust * self.g * self.detachment_depth
    
    @property
    def sigma_effective(self):
        """Effective normal stress after pore pressure reduction (Pa)."""
        return self.sigma_lithostatic * (1 - self.pore_pressure_ratio)
    
    # Motion parameters
    total_distance: float = 1000e3        # m (1000 km total displacement)
    event_duration: float = 3.15e7        # s (1 year)
    
    # Energy partitioning fractions (initial estimates)
    seismic_efficiency: float = 0.10      # Fraction radiated as seismic waves
    plastic_fraction: float = 0.001       # Fraction to plastic deformation
    
    # Earth parameters
    earth_surface_area: float = 5.1e14    # m²
    
    # Water parameters
    rho_water: float = 1000               # kg/m³
    c_water: float = 4186                 # J/(kg·K)
    ocean_mass: float = 1.4e21            # kg


# Create parameter instance
params = ModelParameters()

print("=== Model Parameters ===")
print(f"Block mass: {params.block_mass:.2e} kg")
print(f"Base area: {params.base_area:.2e} m²")
print(f"Lithostatic stress: {params.sigma_lithostatic/1e6:.1f} MPa")
print(f"Effective stress: {params.sigma_effective/1e6:.1f} MPa")
print(f"Stress reduction factor: {(1 - params.pore_pressure_ratio):.2f} (100×)")

## 2. Initial Energy Budget

Calculate the gravitational potential energy available for Stage 2.

In [ ]:
def calculate_initial_energy(params, settling_distance=1000):
    """
    Calculate gravitational PE released when blocks settle.
    
    Parameters:
    -----------
    params : ModelParameters
    settling_distance : float
        Vertical distance blocks settle (m)
    
    Returns:
    --------
    dict : Energy components
    """
    # PE per block from settling
    PE_per_block = params.block_mass * params.g * settling_distance
    
    # Total PE for all blocks
    PE_total = params.n_blocks * PE_per_block
    
    return {
        'PE_per_block': PE_per_block,
        'PE_total': PE_total,
        'settling_distance': settling_distance
    }


# Calculate for different settling scenarios
scenarios = [100, 500, 1000, 2000]  # meters of settling

print("=== Gravitational PE Available ===")
print(f"{'Settling (m)':<15} {'PE/block (J)':<15} {'Total PE (J)':<15}")
print("-" * 45)

for settle in scenarios:
    energy = calculate_initial_energy(params, settle)
    print(f"{settle:<15} {energy['PE_per_block']:<15.2e} {energy['PE_total']:<15.2e}")

# Use 1 km settling as reference
initial_energy = calculate_initial_energy(params, 1000)
PE_total = initial_energy['PE_total']
print(f"\nReference: {PE_total:.2e} J total PE (1 km settling)")

## 3. Frictional Dissipation Model

Calculate friction force and work using Terzaghi effective stress.

In [ ]:
def calculate_friction(params):
    """
    Calculate frictional force and work.
    
    Uses Terzaghi effective stress principle:
    τ = μ × σ'_eff = μ × σ_n × (1 - λ)
    """
    # Shear stress at interface
    tau = params.friction_coeff * params.sigma_effective
    
    # Friction force per block
    F_friction = tau * params.base_area
    
    # Work over total distance (per block)
    W_friction_per_block = F_friction * params.total_distance
    
    # Total frictional work
    W_friction_total = params.n_blocks * W_friction_per_block
    
    return {
        'shear_stress': tau,
        'friction_force': F_friction,
        'work_per_block': W_friction_per_block,
        'work_total': W_friction_total
    }


# Calculate friction
friction = calculate_friction(params)

print("=== Frictional Dissipation (This Model) ===")
print(f"Shear stress τ: {friction['shear_stress']/1e3:.1f} kPa")
print(f"Friction force F: {friction['friction_force']:.2e} N")
print(f"Work per block: {friction['work_per_block']:.2e} J")
print(f"Total frictional work: {friction['work_total']:.2e} J")

# Compare to critic's calculation (no pore pressure)
print("\n=== Critic's Calculation (λ = 0) ===")
tau_critic = params.friction_coeff * params.sigma_lithostatic
F_critic = tau_critic * params.base_area
W_critic = params.n_blocks * F_critic * params.total_distance
print(f"Shear stress τ: {tau_critic/1e6:.1f} MPa")
print(f"Friction force F: {F_critic:.2e} N")
print(f"Total frictional work: {W_critic:.2e} J")

print(f"\n=== Ratio ===")
print(f"Our model / Critic's: {friction['work_total']/W_critic:.4f} ({1/(friction['work_total']/W_critic):.0f}× reduction)")

## 4. Time-Dependent Energy Partitioning

Model how energy flows between different sinks over the course of Stage 2.

In [ ]:
def energy_partitioning_model(t, params, PE_total):
    """
    Calculate cumulative energy in each sink at time t.
    
    Assumes linear release of PE over event duration.
    
    Parameters:
    -----------
    t : float or array
        Time since start (seconds)
    params : ModelParameters
    PE_total : float
        Total gravitational PE available (J)
    
    Returns:
    --------
    dict : Energy in each sink
    """
    # Fraction of event completed
    progress = np.clip(t / params.event_duration, 0, 1)
    
    # PE released so far
    PE_released = PE_total * progress
    
    # Calculate friction work (proportional to distance traveled)
    friction_data = calculate_friction(params)
    W_friction = friction_data['work_total'] * progress
    
    # Seismic radiation (fraction of released PE)
    E_seismic = PE_released * params.seismic_efficiency
    
    # Plastic deformation
    E_plastic = PE_released * params.plastic_fraction
    
    # Viscous dissipation in water (estimate as comparable to friction)
    E_viscous = W_friction * 0.5  # Order of magnitude estimate
    
    # Kinetic energy (small, velocity-dependent)
    # v = distance / time for average velocity
    v_avg = params.total_distance / params.event_duration * progress
    E_kinetic = 0.5 * params.n_blocks * params.block_mass * v_avg**2
    
    # Residual PE (not yet released)
    PE_residual = PE_total - PE_released
    
    # Energy that must thermalize locally (friction + viscous)
    E_local_heat = W_friction + E_viscous
    
    # Energy that thermalizes globally (seismic eventually becomes heat)
    E_global_heat = E_seismic
    
    return {
        'time': t,
        'progress': progress,
        'PE_released': PE_released,
        'PE_residual': PE_residual,
        'E_friction': W_friction,
        'E_seismic': E_seismic,
        'E_plastic': E_plastic,
        'E_viscous': E_viscous,
        'E_kinetic': E_kinetic,
        'E_local_heat': E_local_heat,
        'E_global_heat': E_global_heat
    }


# Test at end of event
final_state = energy_partitioning_model(params.event_duration, params, PE_total)

print("=== Final Energy State (End of Stage 2) ===")
print(f"\nPE Released: {final_state['PE_released']:.2e} J (100%)")
print(f"\nEnergy Sinks:")
print(f"  Friction (local heat):  {final_state['E_friction']:.2e} J ({final_state['E_friction']/PE_total*100:.2f}%)")
print(f"  Seismic radiation:      {final_state['E_seismic']:.2e} J ({final_state['E_seismic']/PE_total*100:.1f}%)")
print(f"  Viscous dissipation:    {final_state['E_viscous']:.2e} J ({final_state['E_viscous']/PE_total*100:.2f}%)")
print(f"  Plastic deformation:    {final_state['E_plastic']:.2e} J ({final_state['E_plastic']/PE_total*100:.2f}%)")
print(f"  Kinetic energy:         {final_state['E_kinetic']:.2e} J ({final_state['E_kinetic']/PE_total*100:.6f}%)")

# Sum check
E_accounted = (final_state['E_friction'] + final_state['E_seismic'] + 
               final_state['E_viscous'] + final_state['E_plastic'] + final_state['E_kinetic'])
print(f"\nTotal accounted: {E_accounted:.2e} J ({E_accounted/PE_total*100:.1f}%)")
print(f"Unaccounted (numerical): {PE_total - E_accounted:.2e} J")

## 5. Time Evolution Plots

In [ ]:
# Create time array (in days for readability)
t_days = np.linspace(0, 365, 366)
t_seconds = t_days * 24 * 3600

# Calculate energy partitioning at each time step
results = [energy_partitioning_model(t, params, PE_total) for t in t_seconds]

# Extract arrays for plotting
E_friction = np.array([r['E_friction'] for r in results])
E_seismic = np.array([r['E_seismic'] for r in results])
E_viscous = np.array([r['E_viscous'] for r in results])
E_plastic = np.array([r['E_plastic'] for r in results])
PE_released = np.array([r['PE_released'] for r in results])

# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Cumulative energy by sink
ax1 = axes[0, 0]
ax1.fill_between(t_days, 0, E_friction/1e23, alpha=0.7, label='Friction (local heat)')
ax1.fill_between(t_days, E_friction/1e23, (E_friction+E_viscous)/1e23, alpha=0.7, label='Viscous dissipation')
ax1.fill_between(t_days, (E_friction+E_viscous)/1e23, (E_friction+E_viscous+E_seismic)/1e23, 
                 alpha=0.7, label='Seismic radiation')
ax1.fill_between(t_days, (E_friction+E_viscous+E_seismic)/1e23, 
                 (E_friction+E_viscous+E_seismic+E_plastic)/1e23, alpha=0.7, label='Plastic deformation')
ax1.plot(t_days, PE_released/1e23, 'k--', linewidth=2, label='PE released')
ax1.set_xlabel('Time (days)')
ax1.set_ylabel('Cumulative Energy (×10²³ J)')
ax1.set_title('Energy Partitioning During Stage 2')
ax1.legend(loc='upper left')
ax1.set_xlim(0, 365)

# Plot 2: Energy fractions
ax2 = axes[0, 1]
final = results[-1]
labels = ['Friction\n(local heat)', 'Seismic\n(global)', 'Viscous', 'Plastic']
sizes = [final['E_friction'], final['E_seismic'], final['E_viscous'], final['E_plastic']]
colors = ['#ff6b6b', '#4ecdc4', '#45b7d1', '#96ceb4']
ax2.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
ax2.set_title(f'Final Energy Distribution\n(Total: {sum(sizes):.2e} J)')

# Plot 3: Heat flux over time
ax3 = axes[1, 0]
# Instantaneous heat flux (derivative of cumulative heat)
dt = t_seconds[1] - t_seconds[0]
E_local_heat = E_friction + E_viscous
heat_rate = np.gradient(E_local_heat, dt)  # J/s = W
heat_flux = heat_rate / params.earth_surface_area  # W/m²

ax3.plot(t_days, heat_flux, 'r-', linewidth=2)
ax3.axhline(y=7, color='g', linestyle='--', label='Model claim (~7 W/m²)')
ax3.axhline(y=600, color='b', linestyle='--', label="Critic's claim (~600 W/m²)")
ax3.set_xlabel('Time (days)')
ax3.set_ylabel('Heat Flux (W/m²)')
ax3.set_title('Local Heat Flux vs Time')
ax3.legend()
ax3.set_ylim(0, 50)

# Plot 4: Comparison bar chart
ax4 = axes[1, 1]
categories = ['This Model\n(with pore pressure)', "Critic's Calc\n(no pore pressure)"]
heat_values = [final['E_friction'] + final['E_viscous'], W_critic]
bars = ax4.bar(categories, np.array(heat_values)/1e23, color=['green', 'red'], alpha=0.7)
ax4.set_ylabel('Frictional Heat (×10²³ J)')
ax4.set_title('Heat Generation Comparison')
for bar, val in zip(bars, heat_values):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
             f'{val:.1e} J', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('energy_partitioning_results.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nAverage heat flux: {np.mean(heat_flux):.1f} W/m²")
print(f"Peak heat flux: {np.max(heat_flux):.1f} W/m²")

## 6. Heat Budget Closure

Verify that all energy is accounted for and calculate temperature effects.

In [ ]:
def calculate_temperature_rise(E_heat, params):
    """
    Calculate temperature rise from heat input.
    
    Parameters:
    -----------
    E_heat : float
        Heat energy (J)
    params : ModelParameters
    
    Returns:
    --------
    dict : Temperature rise estimates
    """
    # If distributed to entire ocean
    dT_ocean = E_heat / (params.ocean_mass * params.c_water)
    
    # If concentrated in water films (10% of surface, 10m thick)
    film_area = 0.1 * params.earth_surface_area
    film_thickness = 10  # m
    film_mass = film_area * film_thickness * params.rho_water
    dT_film = E_heat / (film_mass * params.c_water)
    
    return {
        'dT_ocean': dT_ocean,
        'dT_film': dT_film,
        'film_mass': film_mass
    }


# Calculate for our model
E_local = final_state['E_friction'] + final_state['E_viscous']
temp_our = calculate_temperature_rise(E_local, params)

# Calculate for critic's model
temp_critic = calculate_temperature_rise(W_critic, params)

print("=== Temperature Rise Comparison ===")
print(f"\nThis Model (E_heat = {E_local:.2e} J):")
print(f"  If mixed with ocean: ΔT = {temp_our['dT_ocean']:.3f} K")
print(f"  If in water films:   ΔT = {temp_our['dT_film']:.1f} K")

print(f"\nCritic's Model (E_heat = {W_critic:.2e} J):")
print(f"  If mixed with ocean: ΔT = {temp_critic['dT_ocean']:.1f} K")
print(f"  If in water films:   ΔT = {temp_critic['dT_film']:.0f} K")

print(f"\n=== Survivability Assessment ===")
print(f"Our model: {temp_our['dT_film']:.0f} K local warming - water remains liquid")
print(f"Critic's model: {temp_critic['dT_film']:.0f} K local warming - would vaporize water")

## 7. Sensitivity Analysis

How do results change with different parameter assumptions?

In [ ]:
# Vary pore pressure ratio
lambda_values = np.linspace(0.5, 0.99, 50)
heat_flux_vs_lambda = []

for lam in lambda_values:
    test_params = ModelParameters()
    test_params.pore_pressure_ratio = lam
    friction_test = calculate_friction(test_params)
    avg_flux = friction_test['work_total'] / (params.earth_surface_area * params.event_duration)
    heat_flux_vs_lambda.append(avg_flux)

# Vary friction coefficient
mu_values = np.logspace(-3, -1, 50)
heat_flux_vs_mu = []

for mu in mu_values:
    test_params = ModelParameters()
    test_params.friction_coeff = mu
    friction_test = calculate_friction(test_params)
    avg_flux = friction_test['work_total'] / (params.earth_surface_area * params.event_duration)
    heat_flux_vs_mu.append(avg_flux)

# Plot sensitivity
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.semilogy(lambda_values, heat_flux_vs_lambda, 'b-', linewidth=2)
ax1.axhline(y=7, color='g', linestyle='--', label='Model claim')
ax1.axhline(y=600, color='r', linestyle='--', label='Lethal threshold')
ax1.axvline(x=0.99, color='purple', linestyle=':', label='Model λ = 0.99')
ax1.set_xlabel('Pore Pressure Ratio (λ)')
ax1.set_ylabel('Average Heat Flux (W/m²)')
ax1.set_title('Sensitivity to Pore Pressure Ratio')
ax1.legend()
ax1.set_xlim(0.5, 1.0)

ax2 = axes[1]
ax2.loglog(mu_values, heat_flux_vs_mu, 'b-', linewidth=2)
ax2.axhline(y=7, color='g', linestyle='--', label='Model claim')
ax2.axhline(y=600, color='r', linestyle='--', label='Lethal threshold')
ax2.axvline(x=0.01, color='purple', linestyle=':', label='Model μ = 0.01')
ax2.set_xlabel('Friction Coefficient (μ)')
ax2.set_ylabel('Average Heat Flux (W/m²)')
ax2.set_title('Sensitivity to Friction Coefficient')
ax2.legend()

plt.tight_layout()
plt.savefig('sensitivity_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n=== Critical Parameter Values ===")
print(f"For heat flux < 10 W/m²: λ > {lambda_values[np.searchsorted(heat_flux_vs_lambda[::-1], 10)]:.2f}")
print(f"For heat flux < 100 W/m²: λ > {lambda_values[np.searchsorted(heat_flux_vs_lambda[::-1], 100)]:.2f}")

## 8. Summary and Conclusions

In [ ]:
print("="*60)
print("ENERGY PARTITIONING SIMULATION SUMMARY")
print("="*60)

print(f"\n1. INITIAL ENERGY BUDGET")
print(f"   Total PE available: {PE_total:.2e} J")
print(f"   Source: {params.n_blocks} blocks settling ~1 km")

print(f"\n2. ENERGY PARTITIONING (Final State)")
print(f"   Frictional heating:    {final_state['E_friction']:.2e} J ({final_state['E_friction']/PE_total*100:.2f}%)")
print(f"   Seismic radiation:     {final_state['E_seismic']:.2e} J ({final_state['E_seismic']/PE_total*100:.1f}%)")
print(f"   Viscous dissipation:   {final_state['E_viscous']:.2e} J ({final_state['E_viscous']/PE_total*100:.2f}%)")
print(f"   Plastic deformation:   {final_state['E_plastic']:.2e} J ({final_state['E_plastic']/PE_total*100:.3f}%)")

print(f"\n3. HEAT FLUX")
print(f"   Average (this model):  {np.mean(heat_flux):.1f} W/m²")
print(f"   Critic's calculation:  ~600 W/m²")
print(f"   Reduction factor:      {600/np.mean(heat_flux):.0f}×")

print(f"\n4. TEMPERATURE EFFECTS")
print(f"   Ocean mixing: ΔT = {temp_our['dT_ocean']:.3f} K (negligible)")
print(f"   Water films:  ΔT = {temp_our['dT_film']:.0f} K (survivable)")

print(f"\n5. KEY FINDING")
print(f"   The 100× reduction in heat generation arises from")
print(f"   Terzaghi effective stress with λ = {params.pore_pressure_ratio}")
print(f"   This is standard geotechnical physics, not special pleading.")

print(f"\n6. LIMITATIONS")
print(f"   - Linear PE release assumed (actual may be episodic)")
print(f"   - Seismic efficiency estimated at {params.seismic_efficiency*100:.0f}% (literature: 1-20%)")
print(f"   - Viscous dissipation is order-of-magnitude estimate")
print(f"   - Does not model spatial distribution of heat")

print("\n" + "="*60)